# 06. Analyse a training run

Three utilities under `microrts-agent analysis`:

1. **`metrics`**: per-run PDF plots (WR curve, return, episode length, loss, KL).
   Requires `events.out.tfevents.*` (TensorBoard) for the full set of plots;
   falls back to a partial set if only `eval_results.csv` is present.
2. **`audit`**: sanity-check a run's files. Strictly requires
   `events.out.tfevents.*`.
3. **`params`**: parameter-count table per architecture / per-feature.
   No file dependency.

Plus a bonus section that parses `train.log` directly to plot the WR curve.

**Important about shipped agents under `data/agents/`**: they do NOT include
`events.out.tfevents.*` files (those are archived separately on the GitHub
release `tfevents-agent-archive`, too big for git). Running `metrics` or
`audit` on a shipped agent therefore works only partially or not at all.
The runs you produce yourself (e.g. via [`02_train.ipynb`](02_train.ipynb))
include the TB events and let both commands work fully.

Prereq: [`00_navigate.ipynb`](00_navigate.ipynb) must be green.
For sections 1 and 2 below to be useful, also run
[`02_train.ipynb`](02_train.ipynb) first to produce
`outputs/runs/notebook-train_s1/`.

## 1. `analysis metrics`: PDF plots + textual summary

Generates a `metrics/` subdir under the target run with several PDFs
(WR vs step, return, loss, etc.) and prints a one-line summary.

In [ ]:
import subprocess

from microrts_agent.paths import PROJECT_ROOT

# Use the run produced by 02_train.ipynb (has TB events) when available,
# else fall back to the shipped agent (partial output: no TB plots).
candidate = PROJECT_ROOT / "outputs" / "runs" / "notebook-train_s1"
target = (
    candidate if candidate.exists() else PROJECT_ROOT / "data" / "agents" / "UECD-SingleMap-Best"
)
print(f"Target: {target}\n")

result = subprocess.run(
    ["microrts-agent", "analysis", "metrics", str(target)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout)
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr)

## 2. `analysis audit`: sanity-check a run

Useful after a training run finishes or to validate a shipped agent
before evaluation. Reports missing files, dtype / shape mismatches,
stale path references in the config.

In [ ]:
result = subprocess.run(
    ["microrts-agent", "analysis", "audit", str(target)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=60,
)
print(result.stdout)

## 3. `analysis params`: parameter counts

Prints a table of total parameter count per architecture (and per
feature delta relative to the UNet-Entity-CBAM-Deep baseline). Pure
stdout, no arguments. Useful to remember what each architecture costs
before launching a training run.

In [ ]:
result = subprocess.run(
    ["microrts-agent", "analysis", "params"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=30,
)
print(result.stdout)

## 4. Bonus: parse `train.log` inline and plot the WR curve

The dissertation figure scripts ship a parser at
[`dissertation/figs/figs-python/_data.py`](../dissertation/figs/figs-python/_data.py).
Reusing it gives you a DataFrame with `step, ret, len, wr_<Bot>`
columns, which you can plot however you like. Curves produced here
match the published figures byte-for-byte.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT / "dissertation" / "figs" / "figs-python"))

from _data import parse_train_log, smooth  # noqa: E402

df = parse_train_log("UECD-SingleMap-Best")
print(f"Rows: {len(df)}, columns: {list(df.columns)}\n")
df.head()

In [ ]:
import matplotlib.pyplot as plt

wr_cols = [c for c in df.columns if c.startswith("wr_")]

fig, ax = plt.subplots(figsize=(10, 5))
for col in wr_cols:
    bot = col[len("wr_") :]
    ax.plot(df["step"] / 1e6, smooth(df[col].values, window=20) * 100, label=bot, alpha=0.85)
ax.set_xlabel("Training step (M)")
ax.set_ylabel("Smoothed win rate (%)")
ax.set_title("UECD-SingleMap-Best: per-opponent win rate over training")
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Side-by-side: baseline vs final agent

Quick comparison of `GridNet-SingleMap` (the published baseline) and
`UECD-SingleMap-Best` (the dissertation's headline).

In [ ]:
df_gridnet = parse_train_log("GridNet-SingleMap")
df_best = df  # already loaded above

wr_cols_gridnet = [c for c in df_gridnet.columns if c.startswith("wr_")]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(
    df_gridnet["step"] / 1e6,
    smooth(df_gridnet[wr_cols_gridnet].mean(axis=1).values, window=20) * 100,
    label="GridNet-SingleMap (baseline, 0.84M params)",
    linewidth=2,
    color="#888",
)
ax.plot(
    df_best["step"] / 1e6,
    smooth(df_best[wr_cols].mean(axis=1).values, window=20) * 100,
    label="UECD-SingleMap-Best (final, 4.7M params)",
    linewidth=2,
    color="#1f77b4",
)
ax.set_xlabel("Training step (M)")
ax.set_ylabel("Pool mean win rate (%)")
ax.set_title("Baseline vs final agent: pool mean WR over training")
ax.set_ylim(0, 105)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Next steps

- Run `analysis metrics` on the run produced by
  [`02_train.ipynb`](02_train.ipynb): point at
  `outputs/runs/notebook-train_s1/` instead of the shipped agent.
- TensorBoard: each run also ships `events.out.tfevents.*`. Launch with
  `tensorboard --logdir data/agents/UECD-SingleMap-Best` (you'll see
  loss, KL, value, return curves at full resolution).
- The full thesis figure pipeline is under
  [`dissertation/figs/figs-python/`](../dissertation/figs/figs-python/);
  every script there is re-runnable on a fresh clone.